# PlantSeg Round-2 — Notebook F: ConvNeXtV2-Tiny 78.4% headline-number re-check

**Purpose.** Resolve the open TTA/headline-number discrepancy (R4 Concern 3 / R5 point 4) in one Kaggle run, without another round of back-and-forth.

Context: Table 1's headline classifier number is ConvNeXtV2-Tiny, **78.4%** with simple flip-TTA (test-time augmentation — sum the logits of an image and its horizontal flip, then argmax), vs. **67.6%** single-pass, already disclosed. An independent re-implementation (Notebook A, run in a separate CPU-only environment) loaded the checkpoint currently sitting in the project's public GitHub repo and got only 67.65% single-pass (matches) but just **68.55%** with the identical TTA formula — not 78.4%. Static code review found the likely explanation: the notebook treated as the "canonical" source of 78.4% (`PlantSeg_S1_Session2_Train_Models5to8.ipynb`) still has a real bug for this one model — it never applies the low-LR(5e-5)+warmup treatment that ConvNeXtV2-Tiny actually needs (confirmed by a **separate, standalone notebook, `PlantSeg_S1_5_ConvNeXtV2Tiny_FIXED.ipynb`**, whose own header says: *"ConvNeXtV2-Tiny now trains at LR 5e-5 + warmup — it diverged at 3e-4 in the first run, collapsing to one class"*). That fixed notebook writes to the exact same output filenames, so it may be the real, un-diffed source of the 78.4% result — or the checkpoint currently in the repo may simply be stale relative to whichever run actually produced it.

**What this notebook does, in one GPU session (expect roughly 75–115 minutes based on this exact model's seed-123/seed-7 retraining times: 76 min / 28 epochs and 113 min / 40 epochs respectively):**

- **Part A** — retrains ConvNeXtV2-Tiny from scratch, seed 42, using the *exact* fixed recipe (`PlantSeg_S1_5_ConvNeXtV2Tiny_FIXED.ipynb`'s own `train_one()`, copied verbatim, low LR + warmup applied). Reports this fresh run's own single-pass and TTA test accuracy.
- **Part B** — independently reloads the checkpoint **currently sitting in the repo/dataset you attach** and re-runs the identical TTA formula live on Kaggle (the same thing Notebook A already did off-Kaggle, repeated here to rule out an environment difference).
- **Part C** — prints a single comparison table and an explicit, unambiguous verdict:
  - If Part A's TTA accuracy ≈ 78.4% → the *fixed* recipe really does produce the headline number; the repo's current checkpoint is simply stale (from a different/earlier run) and should be swapped for this notebook's fresh checkpoint.
  - If Part A's TTA accuracy ≈ Part B's (~68%) → 78.4% does not reproduce even from its most likely source; genuine run-to-run variance is the explanation, and Table 1/Abstract should switch to the reproducible number (the 3-seed mean is already the paper's primary headline per the latest revision, so this is a contained change).
  - Any other pattern is printed plainly rather than forced into one of the two stories above.

**Before running:** attach the PlantSeg dataset as a Kaggle input (same as every other notebook in this project), and attach a dataset containing the **current** `ConvNeXtV2Tiny_best.pt` checkpoint (the one currently in `PlantSeg_Deployment_Space/` / the public GitHub repo) — Part B needs it. This can be the same dataset you use for the other models (one dataset with all checkpoints bundled together is fine) — the CONFIG cell below recursively searches every attached `/kaggle/input/...` dataset for a file named `ConvNeXtV2Tiny_best.pt` (or a close-match filename) and picks it up automatically, regardless of the dataset's slug or folder layout. If nothing is found it prints every `.pt` file it *did* see under `/kaggle/input`, so you can set `CURRENT_CKPT_PATH` by hand from that list. Internet **ON** (timm weights download, first run only — the fixed recipe uses ImageNet-pretrained init).


In [1]:
!pip install -q timm
import os, json, math, time, random, warnings, pickle, glob
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                              precision_score, recall_score, matthews_corrcoef,
                              confusion_matrix, roc_auc_score)
warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
assert device.type == "cuda", "This notebook needs a GPU — enable one under Settings before running Part A."


Device: cuda


In [2]:
# ============ CONFIG ============
DATA_ROOT = Path("/kaggle/input/datasets/alenmaruf/plantseg/plantseg")
if not (DATA_ROOT/"Metadata.csv").exists():
    for cand in [Path("/kaggle/input/plantseg"), Path("plantseg"), Path(".")]:
        if (cand/"Metadata.csv").exists(): DATA_ROOT = cand; break
assert (DATA_ROOT/"Metadata.csv").exists(), f"Metadata.csv not found under {DATA_ROOT} — attach the PlantSeg dataset and/or edit DATA_ROOT."

# Where Part B should look for the CURRENT (repo) ConvNeXtV2Tiny_best.pt checkpoint.
# Auto-discovered by searching every attached /kaggle/input dataset recursively, so this
# works regardless of the dataset's slug/mount name or folder structure (e.g. a single
# dataset bundling ALL models' .pt files together in one or more subfolders).
_exact_hits = sorted(glob.glob("/kaggle/input/**/ConvNeXtV2Tiny_best.pt", recursive=True))
if _exact_hits:
    CURRENT_CKPT_PATH = _exact_hits[0]
    if len(_exact_hits) > 1:
        print(f"NOTE: found {len(_exact_hits)} files named ConvNeXtV2Tiny_best.pt — using the first:")
        for h in _exact_hits: print("   ", h)
else:
    # Fallback: filename may differ slightly (case, no "_best", etc.) — search more loosely
    # and print every candidate so you can hand-set CURRENT_CKPT_PATH if the auto-pick is wrong.
    _loose_hits = sorted(set(
        glob.glob("/kaggle/input/**/*onv*ext*2*iny*.pt", recursive=True) +
        glob.glob("/kaggle/input/**/*onvNeXt*2*.pt", recursive=True)
    ))
    CURRENT_CKPT_PATH = _loose_hits[0] if _loose_hits else None
    if _loose_hits:
        print(f"NOTE: no exact 'ConvNeXtV2Tiny_best.pt' match — using closest filename match: {CURRENT_CKPT_PATH}")
        if len(_loose_hits) > 1:
            print("  other candidates found (edit CURRENT_CKPT_PATH by hand if the wrong one was picked):")
            for h in _loose_hits: print("   ", h)

if CURRENT_CKPT_PATH is None:
    print("WARNING: no ConvNeXtV2Tiny checkpoint found anywhere under /kaggle/input.")
    print("All .pt files currently visible under /kaggle/input (for diagnosis):")
    for p in sorted(glob.glob("/kaggle/input/**/*.pt", recursive=True)):
        print("   ", p)
    print("Part B will be skipped unless you set CURRENT_CKPT_PATH by hand to one of the paths above.")
else:
    print("Part B will use checkpoint:", CURRENT_CKPT_PATH)

OUTDIR = Path("/kaggle/working/notebookF_results"); OUTDIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "ConvNeXtV2Tiny"
NUM_EPOCHS = 40; PATIENCE = 8; WEIGHT_DECAY = 1e-4
USE_PRETRAINED = True; USE_CLASS_WEIGHTS = True; LABEL_SMOOTHING = 0.05
IMG_SIZE_CNN = 384; USE_RANDAUG = True; USE_TTA = True
LR_DEFAULT = 3e-4   # what the still-buggy canonical Session2 notebook uses for this model
LR_FIXED   = 5e-5   # what the standalone _FIXED notebook uses for this model (the recipe Part A runs)

# Also attempt to reproduce the ORIGINAL (buggy, divergent) recipe for full confirmation?
# This roughly doubles total runtime (~another 75-115 min) — off by default to keep this a
# single ~1-2h GPU session. Set True only if you want the third data point.
ALSO_REPRODUCE_BUG = False

print("DATA_ROOT:", DATA_ROOT, "| MODEL:", MODEL_NAME, "| ALSO_REPRODUCE_BUG:", ALSO_REPRODUCE_BUG)


Part B will use checkpoint: /kaggle/input/datasets/alenmaruf/model-pt-class-seg-zip/model pt class+Seg/CLassification/43seed_classifier_checkpoints/ConvNeXtV2Tiny_best.pt
DATA_ROOT: /kaggle/input/datasets/alenmaruf/plantseg/plantseg | MODEL: ConvNeXtV2Tiny | ALSO_REPRODUCE_BUG: False


In [3]:
# ============ DATA INDEX (identical convention to every other notebook in this project) ============
meta = pd.read_csv(DATA_ROOT/"Metadata.csv", encoding="utf-8-sig")
meta["split"] = meta["Split"].map({"Training":"train","Validation":"val","Test":"test"})
meta["filepath"] = meta.apply(lambda r: DATA_ROOT/"images"/r["split"]/r["Name"], axis=1)
meta["label"] = meta["Disease"]
meta = meta[meta["filepath"].apply(lambda p: Path(p).exists())].reset_index(drop=True)
CLASS_NAMES = sorted(meta["label"].unique().tolist()); NUM_CLASSES = len(CLASS_NAMES)
label2idx = {c:i for i,c in enumerate(CLASS_NAMES)}
train_df = meta[meta.split=="train"].reset_index(drop=True)
val_df   = meta[meta.split=="val"].reset_index(drop=True)
test_df  = meta[meta.split=="test"].reset_index(drop=True)
counts = train_df["label"].map(label2idx).value_counts().reindex(range(NUM_CLASSES)).fillna(0)
inv = np.where(counts.values>0, 1.0/counts.values, 0.0); cw = inv/inv[inv>0].mean()
class_weights = torch.tensor(cw, dtype=torch.float32, device=device)
print(f"classes={NUM_CLASSES} train={len(train_df)} val={len(val_df)} test={len(test_df)}")
assert len(test_df) == 1561, f"Expected 1561 test images, got {len(test_df)} — check DATA_ROOT/Metadata.csv."


classes=115 train=5367 val=846 test=1561


In [4]:
# ============ TRANSFORMS / DATASET (identical convention) ============
MEAN=[0.485,0.456,0.406]; STD=[0.229,0.224,0.225]
def make_transforms(sz):
    ops=[transforms.RandomResizedCrop(sz,scale=(0.7,1.0)),transforms.RandomHorizontalFlip(),transforms.RandomVerticalFlip(p=0.2)]
    ops+=[transforms.RandAugment(num_ops=2,magnitude=9)] if USE_RANDAUG else [transforms.RandomRotation(15),transforms.ColorJitter(0.2,0.2,0.2,0.05)]
    ops+=[transforms.ToTensor(),transforms.Normalize(MEAN,STD)]
    if USE_RANDAUG: ops+=[transforms.RandomErasing(p=0.25)]
    ev=transforms.Compose([transforms.Resize((sz,sz)),transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
    return transforms.Compose(ops), ev

class DS(Dataset):
    def __init__(s,d,t): s.df=d.reset_index(drop=True); s.t=t
    def __len__(s): return len(s.df)
    def __getitem__(s,i):
        r=s.df.iloc[i]; return s.t(Image.open(r.filepath).convert("RGB")), label2idx[r.label]

_, EVAL_TF = make_transforms(IMG_SIZE_CNN)


In [5]:
# ============ MODEL + METRICS HELPERS ============
def build_convnext(num_classes, pretrained=USE_PRETRAINED):
    return timm.create_model("convnextv2_tiny", pretrained=pretrained, num_classes=num_classes), IMG_SIZE_CNN, 16

def compute_metrics(y, yp, ypb, n):
    cm = confusion_matrix(y, yp, labels=range(n))
    present = [c for c in range(n) if (y==c).sum()>0]; aucs=[]
    for c in present:
        try: aucs.append(roc_auc_score((y==c).astype(int), ypb[:,c]))
        except Exception: pass
    return {"accuracy": accuracy_score(y,yp),
            "balanced_accuracy": balanced_accuracy_score(y,yp),
            "f1_macro": f1_score(y,yp,average="macro"),
            "mcc": matthews_corrcoef(y,yp),
            "roc_auc_macro": float(np.mean(aucs)) if aucs else float("nan")}, cm


In [6]:
# ============ PART A — retrain from scratch with the FIXED recipe (verbatim from
# PlantSeg_S1_5_ConvNeXtV2Tiny_FIXED.ipynb's train_one(), scoped to just this one model) ============
def train_one_fixed(num_epochs=NUM_EPOCHS, patience=PATIENCE, lr=LR_FIXED, warm_ep=3, pat=10, tag="PartA_fixed"):
    model, img_size, bs = build_convnext(NUM_CLASSES); model = model.to(device)
    ttf, etf = make_transforms(img_size)
    tl = DataLoader(DS(train_df,ttf), batch_size=bs, shuffle=True,  num_workers=2, pin_memory=True, drop_last=True)
    vl = DataLoader(DS(val_df,  etf), batch_size=bs, shuffle=False, num_workers=2, pin_memory=True)
    te = DataLoader(DS(test_df,etf), batch_size=bs, shuffle=False, num_workers=2, pin_memory=True)
    opt  = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    warm = torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.1, total_iters=warm_ep)
    cos  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, num_epochs-warm_ep))
    sch  = torch.optim.lr_scheduler.SequentialLR(opt, [warm, cos], milestones=[warm_ep])
    crit = nn.CrossEntropyLoss(weight=class_weights if USE_CLASS_WEIGHTS else None, label_smoothing=LABEL_SMOOTHING)
    scaler = torch.cuda.amp.GradScaler(enabled=device.type=="cuda")
    hist = {"train_loss":[], "val_loss":[], "train_acc":[], "val_acc":[]}; best=-1; best_state=None; bad=0
    t0 = time.time()
    for ep in range(num_epochs):
        model.train(); run=cor=tot=0
        for x,y in tl:
            x,y = x.to(device),y.to(device); opt.zero_grad()
            with torch.cuda.amp.autocast(enabled=device.type=="cuda"):
                out = model(x); loss = crit(out, y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            run += loss.item()*x.size(0); cor += (out.argmax(1)==y).sum().item(); tot += y.size(0)
        trl, tra = run/tot, cor/tot
        model.eval(); vr=vc=vt=0
        with torch.no_grad():
            for x,y in vl:
                x,y = x.to(device),y.to(device); out = model(x); loss = crit(out,y)
                vr += loss.item()*x.size(0); vc += (out.argmax(1)==y).sum().item(); vt += y.size(0)
        vlo, vac = vr/vt, vc/vt; sch.step()
        for k,v in zip(hist, [trl,vlo,tra,vac]): hist[k].append(v)
        print(f"[{tag}] ep{ep+1}/{num_epochs} lr={opt.param_groups[0]['lr']:.1e} trL={trl:.3f} vL={vlo:.3f} trA={tra:.3f} vA={vac:.3f}")
        if vac > best+1e-4:
            best=vac; best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}; bad=0
        else:
            bad += 1
            if ep >= warm_ep and bad >= pat:
                print(f"  early stop @ ep{ep+1}"); break
    train_time = time.time()-t0
    model.load_state_dict(best_state)
    torch.save(best_state, OUTDIR/f"{tag}_{MODEL_NAME}_best.pt")
    model.eval(); Y=[]; P_single=[]; P_tta=[]; PR_tta=[]
    with torch.no_grad():
        for x,y in te:
            x = x.to(device)
            lg_o = model(x)
            lg_f = model(torch.flip(x, dims=[3]))
            lg_tta = lg_o + lg_f
            P_single.extend(lg_o.argmax(1).cpu().numpy())
            p_tta = F.softmax(lg_tta, 1).cpu().numpy()
            P_tta.extend(p_tta.argmax(1)); PR_tta.extend(p_tta)
            Y.extend(y.numpy())
    Y=np.array(Y); P_single=np.array(P_single); P_tta=np.array(P_tta); PR_tta=np.array(PR_tta)
    metrics_tta, cm_tta = compute_metrics(Y, P_tta, PR_tta, NUM_CLASSES)
    acc_single = accuracy_score(Y, P_single)
    return {"tag": tag, "hist": hist, "train_time_sec": train_time, "epochs_run": len(hist["train_acc"]),
            "y_true": Y, "y_pred_tta": P_tta, "y_prob_tta": PR_tta, "y_pred_single": P_single,
            "metrics_tta": metrics_tta, "acc_single": acc_single, "cm_tta": cm_tta}


In [7]:
# ============ RUN PART A ============
print("="*70); print("PART A — fresh retrain, seed 42, FIXED recipe (LR=5e-5 + warmup, pat=10)"); print("="*70)
partA = train_one_fixed(lr=LR_FIXED, warm_ep=3, pat=10, tag="PartA_fixed")
pickle.dump(partA, open(OUTDIR/"partA_fixed_result.pkl", "wb"))
print()
print(f"PART A RESULT — epochs run: {partA['epochs_run']} | train_time: {partA['train_time_sec']/60:.1f} min")
print(f"  single-pass accuracy : {partA['acc_single']*100:.2f}%")
print(f"  TTA accuracy         : {partA['metrics_tta']['accuracy']*100:.2f}%")
print(f"  TTA delta over single-pass: {(partA['metrics_tta']['accuracy']-partA['acc_single'])*100:+.2f} pts")
print(f"  (paper's claimed headline: single-pass 67.6% / TTA 78.4%, a +10.8pt delta)")

if ALSO_REPRODUCE_BUG:
    print(); print("="*70); print("OPTIONAL — reproducing the ORIGINAL (buggy) recipe, LR=3e-4, no warmup/low-LR"); print("="*70)
    partA_bug = train_one_fixed(lr=LR_DEFAULT, warm_ep=2, pat=PATIENCE, tag="PartA_buggy")
    pickle.dump(partA_bug, open(OUTDIR/"partA_buggy_result.pkl", "wb"))
    print(f"BUGGY-RECIPE RESULT — single-pass: {partA_bug['acc_single']*100:.2f}% | TTA: {partA_bug['metrics_tta']['accuracy']*100:.2f}%")
    print("  (the _FIXED notebook's own note claims this recipe diverges/collapses to one class —")
    print("   a result near 1/num_classes ≈ %.2f%% here would confirm that.)" % (100/NUM_CLASSES))


PART A — fresh retrain, seed 42, FIXED recipe (LR=5e-5 + warmup, pat=10)


model.safetensors: reconstructing file:   0%|          |  0.00B /  115MB            

model.safetensors: downloading bytes:           |  0.00B            

[PartA_fixed] ep1/40 lr=2.0e-05 trL=4.963 vL=4.898 trA=0.030 vA=0.061
[PartA_fixed] ep2/40 lr=3.5e-05 trL=3.952 vL=3.325 trA=0.306 vA=0.493
[PartA_fixed] ep3/40 lr=5.0e-05 trL=2.536 vL=2.416 trA=0.586 vA=0.613
[PartA_fixed] ep4/40 lr=5.0e-05 trL=1.844 vL=2.138 trA=0.700 vA=0.689
[PartA_fixed] ep5/40 lr=5.0e-05 trL=1.465 vL=2.003 trA=0.796 vA=0.719
[PartA_fixed] ep6/40 lr=4.9e-05 trL=1.278 vL=1.965 trA=0.845 vA=0.727
[PartA_fixed] ep7/40 lr=4.9e-05 trL=1.178 vL=1.943 trA=0.875 vA=0.733
[PartA_fixed] ep8/40 lr=4.8e-05 trL=1.103 vL=1.896 trA=0.904 vA=0.761
[PartA_fixed] ep9/40 lr=4.7e-05 trL=1.056 vL=1.921 trA=0.919 vA=0.751
[PartA_fixed] ep10/40 lr=4.6e-05 trL=1.027 vL=1.907 trA=0.928 vA=0.751
[PartA_fixed] ep11/40 lr=4.4e-05 trL=0.988 vL=1.896 trA=0.937 vA=0.781
[PartA_fixed] ep12/40 lr=4.3e-05 trL=0.974 vL=1.902 trA=0.943 vA=0.743
[PartA_fixed] ep13/40 lr=4.2e-05 trL=0.964 vL=1.955 trA=0.949 vA=0.758
[PartA_fixed] ep14/40 lr=4.0e-05 trL=0.955 vL=1.945 trA=0.949 vA=0.757
[PartA_fixed] e

In [8]:
# ============ PART B — independently reload the CURRENT (repo) checkpoint and re-run
# the identical TTA formula live on Kaggle (repeats Notebook A's off-Kaggle finding here) ============
def safe_load_state(model, ckpt_path):
    obj = torch.load(ckpt_path, map_location="cpu")
    state = obj.get("state_dict", obj) if isinstance(obj, dict) and "state_dict" in obj else obj
    model_keys = set(model.state_dict().keys()); ckpt_keys = set(state.keys())
    missing = model_keys - ckpt_keys; unexpected = ckpt_keys - model_keys
    print(f"[safe_load_state] missing={len(missing)} unexpected={len(unexpected)} (of {len(model_keys)} model keys)")
    if len(missing) >= 0.5*len(model_keys):
        raise RuntimeError(f"Checkpoint at {ckpt_path} does not structurally match this model "
                            f"({len(missing)}/{len(model_keys)} keys missing) — wrong file or wrong architecture.")
    model.load_state_dict(state, strict=False)
    return missing, unexpected

def eval_checkpoint_tta(ckpt_path, tag="PartB_current_ckpt"):
    model, img_size, bs = build_convnext(NUM_CLASSES, pretrained=False)
    safe_load_state(model, ckpt_path)
    model = model.to(device).eval()
    _, etf = make_transforms(img_size)
    te = DataLoader(DS(test_df, etf), batch_size=bs, shuffle=False, num_workers=2, pin_memory=True)
    Y=[]; P_single=[]; P_tta=[]; PR_tta=[]
    with torch.no_grad():
        for x,y in te:
            x = x.to(device)
            lg_o = model(x); lg_f = model(torch.flip(x, dims=[3])); lg_tta = lg_o + lg_f
            P_single.extend(lg_o.argmax(1).cpu().numpy())
            p_tta = F.softmax(lg_tta, 1).cpu().numpy()
            P_tta.extend(p_tta.argmax(1)); PR_tta.extend(p_tta)
            Y.extend(y.numpy())
    Y=np.array(Y); P_single=np.array(P_single); P_tta=np.array(P_tta); PR_tta=np.array(PR_tta)
    metrics_tta, _ = compute_metrics(Y, P_tta, PR_tta, NUM_CLASSES)
    acc_single = accuracy_score(Y, P_single)
    chance = 1.0/NUM_CLASSES
    if acc_single < 2*chance:
        print(f"  WARNING: single-pass accuracy ({acc_single*100:.2f}%) is near chance ({chance*100:.2f}%) — "
              f"this checkpoint may not match this architecture/class list. Treat results with caution.")
    return {"tag": tag, "ckpt_path": str(ckpt_path), "y_true": Y, "y_pred_tta": P_tta, "y_prob_tta": PR_tta,
            "y_pred_single": P_single, "metrics_tta": metrics_tta, "acc_single": acc_single}


In [9]:
# ============ RUN PART B ============
if CURRENT_CKPT_PATH is not None:
    print("="*70); print("PART B — live re-eval of the CURRENT repo checkpoint on Kaggle"); print("="*70)
    partB = eval_checkpoint_tta(CURRENT_CKPT_PATH, tag="PartB_current_ckpt")
    pickle.dump(partB, open(OUTDIR/"partB_current_ckpt_result.pkl", "wb"))
    print()
    print(f"PART B RESULT — checkpoint: {CURRENT_CKPT_PATH}")
    print(f"  single-pass accuracy : {partB['acc_single']*100:.2f}%  (Notebook A, off-Kaggle, got 67.65%)")
    print(f"  TTA accuracy         : {partB['metrics_tta']['accuracy']*100:.2f}%  (Notebook A, off-Kaggle, got 68.55%)")
else:
    print("Part B SKIPPED — no current checkpoint found. Set CURRENT_CKPT_PATH in the CONFIG cell and re-run cells 3, 9, 10.")
    partB = None


PART B — live re-eval of the CURRENT repo checkpoint on Kaggle
[safe_load_state] missing=0 unexpected=0 (of 200 model keys)

PART B RESULT — checkpoint: /kaggle/input/datasets/alenmaruf/model-pt-class-seg-zip/model pt class+Seg/CLassification/43seed_classifier_checkpoints/ConvNeXtV2Tiny_best.pt
  single-pass accuracy : 67.65%  (Notebook A, off-Kaggle, got 67.65%)
  TTA accuracy         : 68.55%  (Notebook A, off-Kaggle, got 68.55%)


In [10]:
# ============ PART C — comparison table + explicit verdict ============
rows = []
rows.append({"source": "Paper's claimed headline (Table 1 / Abstract)", "single_pass": 0.676, "tta": 0.784, "delta": 0.784-0.676})
rows.append({"source": "Notebook A, off-Kaggle, on CURRENT repo checkpoint", "single_pass": 0.6765, "tta": 0.6855, "delta": 0.6855-0.6765})
if partB is not None:
    rows.append({"source": "Part B (this notebook), live on Kaggle, on CURRENT repo checkpoint",
                 "single_pass": partB["acc_single"], "tta": partB["metrics_tta"]["accuracy"],
                 "delta": partB["metrics_tta"]["accuracy"] - partB["acc_single"]})
rows.append({"source": "Part A (this notebook), fresh retrain with the FIXED recipe",
             "single_pass": partA["acc_single"], "tta": partA["metrics_tta"]["accuracy"],
             "delta": partA["metrics_tta"]["accuracy"] - partA["acc_single"]})
if ALSO_REPRODUCE_BUG:
    rows.append({"source": "Part A (optional), fresh retrain with the ORIGINAL buggy recipe",
                 "single_pass": partA_bug["acc_single"], "tta": partA_bug["metrics_tta"]["accuracy"],
                 "delta": partA_bug["metrics_tta"]["accuracy"] - partA_bug["acc_single"]})

cmp_df = pd.DataFrame(rows)
cmp_df["single_pass_%"] = (cmp_df["single_pass"]*100).round(2)
cmp_df["tta_%"] = (cmp_df["tta"]*100).round(2)
cmp_df["delta_pts"] = (cmp_df["delta"]*100).round(2)
print(cmp_df[["source","single_pass_%","tta_%","delta_pts"]].to_string(index=False))
cmp_df.to_csv(OUTDIR/"notebookF_comparison_table.csv", index=False)

print()
print("="*70); print("VERDICT"); print("="*70)
paper_tta = 0.784
a_tta = partA["metrics_tta"]["accuracy"]
b_tta = partB["metrics_tta"]["accuracy"] if partB is not None else None

if abs(a_tta - paper_tta) <= 0.01:
    verdict = ("Part A's fresh retrain with the FIXED recipe reproduces the paper's 78.4% headline "
               "(within 1pt). The FIXED notebook IS the real source of the reported number. "
               "The checkpoint currently in the repo is stale relative to that run — "
               f"replace it with this notebook's {OUTDIR}/PartA_fixed_{MODEL_NAME}_best.pt "
               "and the discrepancy is fully explained: a file-provenance gap, not a research-integrity problem.")
elif b_tta is not None and abs(a_tta - b_tta) <= 0.02:
    verdict = ("Part A's fresh retrain does NOT reproduce 78.4% — it lands close to Part B's "
               "~68% instead. 78.4% does not reproduce even from its most likely source notebook. "
               "This is genuine run-to-run variance (same shape as the ViT/Swin seed-42 finding "
               "earlier in this revision). Recommended fix: switch Table 1 / Abstract / Highlights / "
               "the related-work comparison table's ConvNeXtV2-Tiny single-seed figure to the "
               f"reproducible {a_tta*100:.1f}% TTA / {partA['acc_single']*100:.1f}% single-pass numbers "
               "from this run, and note that the 3-seed mean (already the paper's primary classification "
               "headline per the latest revision) is unaffected by this single-seed figure's correction.")
else:
    verdict = ("Neither clean pattern holds: Part A's result doesn't match 78.4%, and doesn't match "
               "Part B's ~68% either. This is a genuinely new pattern — do not force it into either "
               "story above. Report the raw numbers in notebookF_comparison_table.csv as-is and "
               "treat this as still open pending further investigation.")
print(verdict)

summary = {"paper_claimed_tta": paper_tta, "paper_claimed_single": 0.676,
           "notebookA_offkaggle_tta": 0.6855, "notebookA_offkaggle_single": 0.6765,
           "partB_live_tta": b_tta, "partB_live_single": (partB["acc_single"] if partB is not None else None),
           "partA_fresh_retrain_tta": a_tta, "partA_fresh_retrain_single": partA["acc_single"],
           "partA_epochs_run": partA["epochs_run"], "partA_train_time_sec": partA["train_time_sec"],
           "verdict": verdict}
json.dump(summary, open(OUTDIR/"notebookF_SUMMARY.json","w"), indent=2)
print(); print("Saved:", OUTDIR/"notebookF_SUMMARY.json", "and", OUTDIR/"notebookF_comparison_table.csv")
print("Also saved per-run result pickles and the fresh checkpoint under", OUTDIR)


                                                            source  single_pass_%  tta_%  delta_pts
                     Paper's claimed headline (Table 1 / Abstract)          67.60  78.40       10.8
                Notebook A, off-Kaggle, on CURRENT repo checkpoint          67.65  68.55        0.9
Part B (this notebook), live on Kaggle, on CURRENT repo checkpoint          67.65  68.55        0.9
       Part A (this notebook), fresh retrain with the FIXED recipe          76.87  76.87        0.0

VERDICT
Neither clean pattern holds: Part A's result doesn't match 78.4%, and doesn't match Part B's ~68% either. This is a genuinely new pattern — do not force it into either story above. Report the raw numbers in notebookF_comparison_table.csv as-is and treat this as still open pending further investigation.

Saved: /kaggle/working/notebookF_results/notebookF_SUMMARY.json and /kaggle/working/notebookF_results/notebookF_comparison_table.csv
Also saved per-run result pickles and the fresh check